In [22]:
import re
import json

In [ ]:
class BPEMethod:
    def __init__(self, file_path: str):
        with open(file_path) as file:
            self.text = self.format_text(file.read())

    def format_text(self, text: str):
        text = text.lower()
        text = re.sub(r'[^а-яёa-z0-9 ,.!?-]', '', text)
        text = re.sub(r' +', ' ', text)

        return text

    def get_counts(self):
        counts = {}
        for a, b in zip(self.text, self.text[1:]):
            key = (a, b)
            if key not in counts:
                counts[key] = 1
            else:
                counts[key] += 1
        return counts

    def check_bytes(self, codes: dict):
        format_codes = {k: f'{v[0]}{v[1]}' for k, v in codes.items()}
        json_string = json.dumps(format_codes, separators=[':', ','])
        return len(json_string.encode('utf-8')) <= 1024

    def replace_pair(self, index: int, pair: str):
        str_pair = ''.join(str(v) for v in pair)
        self.text = self.text.replace(str_pair, str(index))

    def bpe_compress(self):

        codes = {}

        while True:
            counts = self.get_counts()
            if not counts or max(counts.values()) == 1:
                break
            
            max_pair = max(counts, key=counts.get)
            last_key = max(codes) if codes else -1

            test_codes = codes.copy()
            test_codes[last_key + 1] = max_pair
            if not self.check_bytes(test_codes):
                break

            codes[last_key + 1] = max_pair
            self.replace_pair(index=last_key+1, pair=max_pair)
        
        return codes
        

In [69]:
test = BPEMethod('text.txt')
print(test.get_counts())

{('м', 'и'): 504, ('и', 'х'): 192, ('х', 'а'): 71, ('а', 'и'): 19, ('и', 'л'): 669, ('л', ' '): 811, (' ', 'б'): 692, ('б', 'у'): 115, ('у', 'л'): 257, ('л', 'г'): 14, ('г', 'а'): 199, ('а', 'к'): 479, ('к', 'о'): 1203, ('о', 'в'): 1337, ('в', '.'): 46, ('.', 'р'): 9, ('р', 'о'): 1115, ('о', 'к'): 341, ('в', 'ы'): 478, ('ы', 'е'): 207, ('е', ' '): 1557, (' ', 'я'): 201, ('я', 'й'): 46, ('й', 'ц'): 45, ('ц', 'а'): 119, ('а', 'г'): 126, ('г', 'л'): 234, ('л', 'а'): 736, ('а', 'в'): 315, ('в', 'а'): 784, ('а', ' '): 1581, (' ', '1'): 53, ('1', '.'): 2, ('.', 'к'): 22, ('к', 'у'): 299, ('у', 'р'): 174, ('р', 'р'): 12, ('р', 'и'): 602, ('и', 'к'): 506, ('л', 'ю'): 84, ('ю', 'м'): 4, ('м', ' '): 702, (' ', 'в'): 2020, ('в', 'и'): 458, ('и', 'т'): 407, ('т', 'э'): 2, ('э', ' '): 1, (' ', 'п'): 2336, ('п', 'р'): 849, ('о', 'ф'): 173, ('ф', 'е'): 172, ('е', 'с'): 650, ('с', 'с'): 231, ('с', 'о'): 486, ('о', 'р'): 780, ('р', 'а'): 1054, ('п', 'е'): 520, ('е', 'р'): 1136, ('р', 'с'): 283, ('с', '

In [70]:
print(test.bpe_compress())

{0: (',', ' '), 1: (' ', 'п'), 2: ('о', ' '), 3: ('и', ' '), 4: ('а', ' '), 5: ('е', ' '), 6: ('о', 'в'), 7: ('с', 'т'), 8: ('а', 'л'), 9: ('е', 'р'), 10: ('р', 'о'), 11: ('1', '0'), 12: ('1', '1'), 13: ('1', '2'), 14: ('1', '3'), 15: ('1', '4'), 16: ('1', '5'), 17: ('1', '6'), 18: ('1', '7'), 19: ('1', '8'), 20: ('1', '9'), 21: ('2', '0'), 22: ('2', '1'), 23: ('2', '2'), 24: ('2', '3'), 25: ('2', '4'), 26: ('2', '5'), 27: ('2', '6'), 28: ('2', '7'), 29: ('2', '8'), 30: ('2', '9'), 31: ('3', '0'), 32: ('3', '1'), 33: ('3', '2'), 34: ('3', '3'), 35: ('3', '4'), 36: ('3', '5'), 37: ('3', '6'), 38: ('3', '7'), 39: ('3', '8'), 40: ('3', '9'), 41: ('4', '0'), 42: ('4', '1'), 43: ('4', '2'), 44: ('4', '3'), 45: ('4', '4'), 46: ('4', '5'), 47: ('4', '6'), 48: ('4', '7'), 49: ('4', '8'), 50: ('4', '9'), 51: ('5', '0'), 52: ('5', '1'), 53: ('5', '2'), 54: ('5', '3'), 55: ('5', '4'), 56: ('5', '5'), 57: ('5', '6'), 58: ('5', '7'), 59: ('5', '8'), 60: ('5', '9'), 61: ('6', '0'), 62: ('6', '1'), 6

In [71]:
print(test.text)

михаил булгак6.95к6ы5яйцаглав95.куррикулюм витэ951фессора95сик6а95 апреля 9595 года0веч9ом0п95фессор зоологи3iv государ7венног2унив9ситет95директор зооин7итут4в москве95сик6 вошел в свой кабинет0помещающийся в зооин7итуте0чт2н4улиц5г9цена.951фессор зажег в9хний мат6ый шар 3огляделся.нач95ужасающей ката957фы нужн2считать з8оженным именн2в этот злосча7ный веч95равн2как95вопричиною этой ката957фы следует считать именно951фессор4владимир4ипатьевича95сик6а.ему был2р6н958 лет. гол95замечательная0толкачом0лысая0с1учкам3желт6атых волос0торчащими95бокам. лиц2гладк2выбритое0нижняя губ4выпячен4вп9ед. от этого95сик6ско5лиц2вечн2носил2н4себ5нескольк2капризный отпечаток. н4красном носу 7а95модны5м8еньки5очк3в с9ебряной оправе0глазк3бле7ящие0небольшие0957у высокого0сутул6ат. г6орил скрипучим0тонким0квакающим голосом 3сред3других 7ранно7ей имел такую когд4г6орил что-либ2веск95ув9енно0указательный95ец1равой руки1ревращ8 в крючок 3щурил глазки. 4так как он г6орил всегд4ув9енно0иб2эрудиция в ег2обла95у н